# Tracer bullet — does the architecture hold?

A **tracer bullet**, not the product. It fires the thinnest end-to-end slice that
proves the chain works before v0.1.0 is built:

```
canonical records -> turn -> view -> teacher label -> majority vote
                  -> embedding -> logistic head -> session-disjoint eval -> metrics
```

Runs on a free Colab **CPU** runtime, with **no API key** and **no GPU**. Total
wall time is dominated by the first sentence-transformers model download (~90 MB).

**What it deliberately omits** (all v0.1.0 plan items): adapters, Parquet/DuckDB,
the real teacher, sampling-arm comparison, calibration, abstention, ONNX export,
the ship rule, and the `unmapped` discovery loop.

**One thing it cannot measure:** agreement across sampling arms. The teacher here
is a deterministic mock, so repeated passes are identical by construction and
$\alpha$ would be a meaningless 1.0. Arm selection needs a real teacher.


## 1. Environment

Colab ships `torch`/`transformers`/`pandas` at its own pinned versions. We install into **this** runtime so the result matches local/CI rather than Colab's defaults — silent version skew is exactly the class of difference the plan's `manifest.json` version recording exists to catch.

In [ ]:
# Colab-only bootstrap. Guarded so the notebook also runs locally unchanged.
import sys, subprocess, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "sentence-transformers", "scikit-learn", "pandas", "pyyaml"],
                   check=True)
# The stack is pinned to <3.14: no torch wheel exists for 3.14, so an
# unpinned resolve installs a torch-free environment and every metric below
# would describe a different stack than the one CI pins. Fail loudly.
assert sys.version_info[:2] >= (3, 11) and sys.version_info[:2] < (3, 14), (
    f"python {sys.version.split()[0]} outside the supported range >=3.11,<3.14"
)
print("python", sys.version.split()[0])


## 2. Clone and install

The repo is public, so Colab can clone it directly. This is the same clone-and-run path any stranger would follow, which is the point — a tracer that only works on the author's machine proves nothing.

In [ ]:
REPO = "https://github.com/evanokeefe39/agent-turn-classifier.git"
if IN_COLAB and not os.path.isdir("/content/agent-turn-classifier"):
    subprocess.run(["git", "clone", "-q", REPO, "/content/agent-turn-classifier"], check=True)

# Resolve the repo root ABSOLUTELY, then chdir once. Doing it the other way
# round (chdir to a relative path, then insert a relative "src") makes the
# import depend on the kernel's cwd, which is how this cell first failed under
# nbconvert with ModuleNotFoundError.
ROOT = "/content/agent-turn-classifier" if IN_COLAB else os.path.abspath(
    os.path.join(os.getcwd(), "..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
)
os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))
print("cwd:", os.getcwd())
print("src on path:", os.path.join(ROOT, "src"))
print(sorted(os.listdir(".")))


## 3. Validate the ontology first

Fail loudly before doing any work: a taxonomy that does not validate makes every downstream number meaningless. Note the anti-vacuity rule — an `excludes` reason must be ≥ 4 words, so a placeholder that validates while disambiguating nothing is rejected.

In [ ]:
from agent_turn_classifier import tracer as T

workflows, domains = T.load_ontology("examples/ontology.example.yaml")
errors = T.validate(workflows, domains)
print(f"{len(workflows)} workflows across {len(domains)} domains")
for w in workflows:
    print(f"  {w.id:10} {w.domain:18} {w.name}")
print()
print("validator errors:", errors if errors else "NONE")
assert not errors, "ontology must validate before anything downstream runs"


## 4. Records → turns → view

The view is the single text both teacher and student see. Redaction is applied *before* the hash is computed, so the hash attests the redacted payload — the egress seam's contract.

In [ ]:
turns = T.build_turns(T.read_canonical("examples/sessions.sample.jsonl"))
print(f"{len(turns)} turns from {len({t.session_id for t in turns})} sessions\n")

t0 = turns[0]
print(T.render(t0))
print("\nview sha256:", T.view_sha(T.render(t0)))


## 5. Teacher labels → majority vote

The mock reads canned labels keyed on the **rendered-view hash**, never the truth file. That matters: a mock that read back the answer would make the eval below vacuous.

In [ ]:
import json
from pathlib import Path
from collections import Counter

canned = {json.loads(l)["view_sha256"]: json.loads(l)["workflow"]
          for l in Path("examples/canned_labels.jsonl").read_text().splitlines() if l.strip()}
teacher = T.MockTeacher({T.render(t): canned.get(T.view_sha(T.render(t)), "unmapped") for t in turns})

majority_rows = {}
for t in turns:
    votes = [teacher.label(t, T.render(t)) for _ in range(3)]   # the plan's 3-pass arm
    majority_rows[t.turn_id] = T.majority(votes)

print("label distribution:", dict(Counter(r["workflow"] for r in majority_rows.values())))
print("a majority row:", json.dumps(next(iter(majority_rows.values())), indent=2))


## 6. Session-disjoint folds

Turns inside a session are correlated, so a random split leaks. The tracer **asserts** disjointness on the fitted folds rather than assuming it — the plan's P9 preflight does the same, on the grounds that intent is not evidence.

In [ ]:
views = {t.turn_id: T.render(t) for t in turns}
wf_domain = {w.id: w.domain for w in workflows}
path = lambda wf: f"{wf_domain[wf]}/{wf}" if wf in wf_domain else "unmapped"
labels = {k: path(v["workflow"]) for k, v in majority_rows.items()}

frame = T.build_frame(turns, views, labels)
print(f"{len(frame)} labelled turns, {frame['session_id'].nunique()} sessions, {frame['label'].nunique()} classes")
print()
print(frame.groupby("session_id")["label"].apply(list).to_string())


## 7. Train and evaluate

The student is an encoder plus a multinomial logistic head — the plan's default, chosen because labels-per-class here are tens-to-hundreds from weak labelling, not a few-shot regime. Primary metric is path macro-F1 over the flat `domain/workflow` label.

In [ ]:
import time, json
t0 = time.time()
metrics = T.run_tracer(folds=3)
metrics["wall_seconds"] = round(time.time() - t0, 1)
print(json.dumps(metrics, indent=2))


## 8. Read the result — carefully

**This tracer proves the plumbing, not the numbers.** A diagnostic over
{GroupKFold, LeaveOneOut} x {target=teacher, target=truth} shows why:

```
                    target=teacher   target=truth
  GroupKFold(3)        0.705          0.982
  LeaveOneOut          0.691          0.966
```

- **No leakage.** Student-vs-truth is 0.982 (GroupKFold) vs 0.966 (LOO). LOO
  trains on *more* turns (71 vs 48), so leakage would raise it — it does not.
  All 72 nearest neighbours are same-workflow, 0 cross-workflow: the high cosine
  is class signal, not duplicated text.
- **The target column explains the 0.26.** Scoring against teacher labels is
  harder because the mock's labels carry an injected ~26% error rate. Nothing
  to do with the split.
- **`teacher_path_macro_f1` is not a ceiling.** The mock reads a canned file, so
  it is exactly `1 - injected_error_rate` and bounds nothing.
- **`ceiling_gap` is therefore NEGATIVE**, which cannot happen against a real
  ceiling. This fixture is simply too easy to measure distillation: six
  synthetic workflows, 12 examples per class, separable by bge-small.

**Plan-level consequence, the thing worth carrying forward:** `ceiling_gap`
cannot detect distillation failure when a student can exceed its teacher — and
the ship rule keys on exactly that quantity. The ship rule needs a different
gate (teacher-vs-truth on held-out turns, or a corpus where the teacher's
accuracy genuinely exceeds the student's).

What the tracer *does* prove:

1. The chain runs end to end offline on a free CPU runtime, no key, ~30s.
2. Folds are session-disjoint (asserted on the fitted folds).
3. Teacher and student are scored on the same turn population, and the counts
   partition the turn set — the units are right even where the magnitudes are
   uninformative.

A real ceiling needs a real teacher and a gold set. That is the first spike.


In [ ]:
# The assertions that make this a tracer rather than a demo.
assert metrics["n_turns"] >= 20, "corpus below the density the design assumes"
assert metrics["n_sessions"] >= 3, "too few sessions for session-disjoint folds"
assert metrics["n_classes"] >= 3, "too few classes to be a meaningful path metric"
assert 0.0 <= metrics["student_path_macro_f1"] <= 1.0

# Population accounting: the four counts must partition the turn set. This is
# the invariant the tracer actually proves.
assert (
    metrics["n_none"] + metrics["n_unmapped"] + metrics["n_scored"]
    == metrics["n_teacher_labelled"]
), "reserved and scored turns must partition the labelled set"
assert metrics["n_work"] == metrics["n_teacher_labelled"] - metrics["n_none"]
assert metrics["n_scored"] <= metrics["n_work"]

# A student beating its teacher is an ANOMALY: it means the task is easier than
# the teacher's labelling, so the gap measures nothing. This is EXPECTED on the
# synthetic fixture (six separable workflows), so it warns rather than fails —
# but it must never pass silently, because on real data it would mean the
# ship-rule gate is not measuring what it claims to.
gap = metrics["ceiling_gap"]
if metrics["student_path_macro_f1"] > metrics["teacher_path_macro_f1"]:
    print(
        f"WARNING: student ({metrics['student_path_macro_f1']}) exceeds teacher "
        f"({metrics['teacher_path_macro_f1']}); gap={gap}. Expected on the "
        "synthetic fixture — ceiling_gap measures nothing here. See the README."
    )
else:
    assert gap >= 0, f"negative gap {gap} without the student exceeding the teacher"
print("tracer OK: chain runs, folds are session-disjoint, populations partition")
